In [61]:

import pandas as pd
import numpy as np
applicants = pd.read_csv("applicants_cleaned.csv")

# Rename columns to match the assignment
applicants = applicants.rename(columns={
    "Applicant Name": "name",
    "Domain Applied": "domain",
    "University": "university",
    "Application Date": "application_date",
    "Status": "status"
})

np.random.seed(42)
applicants["status"] = np.random.choice(
    ["Under Review", "Selected", "Rejected"],
    size=len(applicants),
    p=[0.20, 0.45, 0.35]   # 20% Under Review, 45% Selected, 35% Rejected
)


# Remove Email and Phone columns
applicants = applicants.drop(columns=["Email", "Phone"])

# Add applicant_id
applicants.insert(0, "applicant_id", range(1, len(applicants) + 1))

# Save the cleaned applicants table
applicants.to_csv("applicants.csv", index=False)

# Preview
applicants.head()

,applicant_id,name,domain,university,application_date,status
0,1,Bob Davis,Web Development,Columbia,2021-04-02,Selected
1,2,Bob Brown,Data Science,Harvard University,2020-07-10,Rejected
2,3,Eva Davis,Cybersecurity,Yale,2021-11-27,Rejected
3,4,Frank Williams,Cloud Computing,Cambridge,2022-01-05,Selected
4,5,Alice Garcia,Web Development,Cambridge,2020-06-10,Under Review


In [62]:
import numpy as np

np.random.seed(42)

selected = applicants[applicants["status"] == "Selected"].copy()

interns = pd.DataFrame({
    "intern_id": range(1001, 1001 + len(selected)),
    "applicant_id": selected["applicant_id"],
    "domain": selected["domain"],
    "start_date": pd.to_datetime(selected["application_date"]) + pd.Timedelta(days=30),
    "completion_status": np.random.choice(
        ["Completed", "Dropped Out"],
        len(selected),
        p=[0.8, 0.2]
    )
})

# Save the cleaned applicants table
interns.to_csv("interns.csv", index=False)
#preview

interns.head()

,intern_id,applicant_id,domain,start_date,completion_status
0,1001,1,Web Development,2021-05-02,Completed
3,1002,4,Cloud Computing,2022-02-04,Dropped Out
8,1003,9,Cloud Computing,2024-08-05,Completed
13,1004,14,Web Development,2024-05-23,Completed
16,1005,17,Cybersecurity,2020-03-02,Completed


In [63]:
completed = interns[interns["completion_status"]=="Completed"].copy()

hackathon_scores = pd.DataFrame({
    "intern_id": completed["intern_id"],
    "score": np.random.randint(60,101,len(completed)),
    "domain": completed["domain"]
})
# Save the cleaned applicants table
hackathon_scores.to_csv("hackathon_scores.csv", index=False)
# preview
hackathon_scores.head()

,intern_id,score,domain
0,1001,62,Web Development
8,1003,96,Cloud Computing
13,1004,66,Web Development
16,1005,80,Cybersecurity
17,1006,68,Web Development


In [64]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("nextgen.db")
applicants.to_sql("applicants", conn, if_exists="replace", index=False)
interns.to_sql("interns", conn, if_exists="replace", index=False)
hackathon_scores.to_sql("hackathon_scores", conn, if_exists="replace", index=False)

conn.commit()

In [65]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';",conn)

,name
0,applicants
1,interns
2,hackathon_scores


In [66]:
pd.read_sql("SELECT * FROM applicants LIMIT 5;",conn)

,applicant_id,name,domain,university,application_date,status
0,1,Bob Davis,Web Development,Columbia,2021-04-02,Selected
1,2,Bob Brown,Data Science,Harvard University,2020-07-10,Rejected
2,3,Eva Davis,Cybersecurity,Yale,2021-11-27,Rejected
3,4,Frank Williams,Cloud Computing,Cambridge,2022-01-05,Selected
4,5,Alice Garcia,Web Development,Cambridge,2020-06-10,Under Review


In [67]:
pd.read_sql("SELECT * FROM interns LIMIT 5;",conn)

,intern_id,applicant_id,domain,start_date,completion_status
0,1001,1,Web Development,2021-05-02 00:00:00,Completed
1,1002,4,Cloud Computing,2022-02-04 00:00:00,Dropped Out
2,1003,9,Cloud Computing,2024-08-05 00:00:00,Completed
3,1004,14,Web Development,2024-05-23 00:00:00,Completed
4,1005,17,Cybersecurity,2020-03-02 00:00:00,Completed


In [68]:
pd.read_sql("SELECT * FROM hackathon_scores LIMIT 5;",conn)

,intern_id,score,domain
0,1001,62,Web Development
1,1003,96,Cloud Computing
2,1004,66,Web Development
3,1005,80,Cybersecurity
4,1006,68,Web Development


In [69]:
#-- Counts completed interns grouped by domain
query1 = """
SELECT
domain,
COUNT(*) AS completed_count
FROM interns
WHERE completion_status = 'Completed'
GROUP BY domain
ORDER BY completed_count DESC;
"""
pd.read_sql(query1, conn)

,domain,completed_count
0,Web Development,12
1,Data Science,5
2,Cloud Computing,3
3,Machine Learning,2
4,Cybersecurity,2


In [70]:
query2 = """
SELECT
    domain,
    ROUND(AVG(score),2) AS avg_score
FROM hackathon_scores
GROUP BY domain
ORDER BY avg_score DESC;
"""

pd.read_sql(query2, conn)

,domain,avg_score
0,Cloud Computing,90.0
1,Machine Learning,83.5
2,Data Science,83.0
3,Cybersecurity,71.5
4,Web Development,70.5


In [71]:
query3 = """
SELECT
    i.intern_id,
    i.domain,
    h.score
FROM interns i
JOIN hackathon_scores h
ON i.intern_id = h.intern_id
WHERE h.score >= 85
ORDER BY h.score DESC;
"""

pd.read_sql(query3, conn)

,intern_id,domain,score
0,1026,Web Development,99
1,1007,Data Science,98
2,1003,Cloud Computing,96
3,1025,Cloud Computing,95
4,1022,Machine Learning,94
5,1019,Data Science,87
6,1016,Data Science,85


In [72]:
query4 = """
SELECT
    a.domain,
    COUNT(DISTINCT a.applicant_id) AS total_applicants,
    COUNT(DISTINCT i.intern_id) AS total_completed,
    ROUND(
        100.0 * COUNT(DISTINCT i.intern_id) /
        COUNT(DISTINCT a.applicant_id),2
    ) AS conversion_rate_pct
FROM applicants a
LEFT JOIN interns i
ON a.applicant_id = i.applicant_id
AND i.completion_status='Completed'
GROUP BY a.domain
ORDER BY conversion_rate_pct DESC;
"""

pd.read_sql(query4, conn)

,domain,total_applicants,total_completed,conversion_rate_pct
0,Web Development,21,12,57.14
1,Data Science,14,5,35.71
2,Machine Learning,7,2,28.57
3,Cloud Computing,12,3,25.00
4,Cybersecurity,11,2,18.18


In [73]:
query5 = """
SELECT
    a.university,
    ROUND(AVG(h.score),2) AS average_score,
    COUNT(h.intern_id) AS total_interns
FROM applicants a
JOIN interns i
ON a.applicant_id = i.applicant_id
JOIN hackathon_scores h
ON i.intern_id = h.intern_id
GROUP BY a.university
ORDER BY average_score DESC;
"""

pd.read_sql(query5, conn)

,university,average_score,total_interns
0,Not Specified,99.00,1
1,Oxford,96.00,1
2,Harvard University,85.71,7
3,Stanford,76.00,3
4,Yale,69.75,4
5,Chicago,69.50,2
6,Princeton,69.00,3
7,MIT,68.00,1
8,UC Berkeley,63.00,1
9,Columbia,62.00,1


In [74]:
!pip install streamlit

In [75]:
%%writefile dashboard.py
import streamlit as st
import pandas as pd
import sqlite3

# -----------------------------------
# Page Configuration
# -----------------------------------
st.set_page_config(
    page_title="NextGenLearners Dashboard",
    page_icon="📊",
    layout="wide"
)

st.title("📊 NextGenLearners Program Performance Dashboard")

# -----------------------------------
# Load Database
# -----------------------------------
conn = sqlite3.connect("nextgen.db")

applicants = pd.read_sql("SELECT * FROM applicants;", conn)
interns = pd.read_sql("SELECT * FROM interns;", conn)
scores = pd.read_sql("SELECT * FROM hackathon_scores;", conn)

# -----------------------------------
# Domain Filter
# -----------------------------------
domain_list = ["All"] + sorted(applicants["domain"].unique().tolist())

selected_domain = st.selectbox(
    "Filter by Domain",
    domain_list
)

# -----------------------------------
# Filter Data
# -----------------------------------
if selected_domain == "All":

    applicants_filtered = applicants

    interns_filtered = interns

    scores_filtered = scores

else:

    applicants_filtered = applicants[
        applicants["domain"] == selected_domain
    ]

    interns_filtered = interns[
        interns["domain"] == selected_domain
    ]

    scores_filtered = scores[
        scores["domain"] == selected_domain
    ]

# -----------------------------------
# Program Funnel
# -----------------------------------
st.subheader("Program Funnel")

col1, col2 = st.columns(2)

with col1:
    st.metric(
        "Total Applicants",
        len(applicants_filtered)
    )

with col2:

    completed = len(
        interns_filtered[
            interns_filtered["completion_status"] == "Completed"
        ]
    )

    st.metric(
        "Completed Interns",
        completed
    )

# -----------------------------------
# Completion Rate per Domain
# -----------------------------------
st.subheader("Completion Rate")

query4 = """
SELECT
    a.domain,
    COUNT(DISTINCT a.applicant_id) AS total_applicants,
    COUNT(DISTINCT i.intern_id) AS total_completed,
    ROUND(
        100.0 * COUNT(DISTINCT i.intern_id) /
        COUNT(DISTINCT a.applicant_id),2
    ) AS conversion_rate_pct
FROM applicants a
LEFT JOIN interns i
ON a.applicant_id = i.applicant_id
AND i.completion_status='Completed'
GROUP BY a.domain
ORDER BY conversion_rate_pct DESC;
"""

conversion = pd.read_sql(query4, conn)

if selected_domain != "All":
    conversion = conversion[
        conversion["domain"] == selected_domain
    ]

st.bar_chart(
    conversion.set_index("domain")[
        "conversion_rate_pct"
    ]
)

# -----------------------------------
# Average Score per Domain
# -----------------------------------
st.subheader("Average Hackathon Score")

query2 = """
SELECT
    domain,
    ROUND(AVG(score),2) AS avg_score
FROM hackathon_scores
GROUP BY domain
ORDER BY avg_score DESC;
"""

avg_score = pd.read_sql(query2, conn)

if selected_domain != "All":
    avg_score = avg_score[
        avg_score["domain"] == selected_domain
    ]

st.bar_chart(
    avg_score.set_index("domain")[
        "avg_score"
    ]
)

# -----------------------------------
# Top 10 Performers
# -----------------------------------
st.subheader("Top 10 Performers")

query3 = """
SELECT
    i.intern_id,
    a.name,
    i.domain,
    h.score
FROM interns i
JOIN applicants a
ON i.applicant_id = a.applicant_id
JOIN hackathon_scores h
ON i.intern_id = h.intern_id
ORDER BY h.score DESC;
"""

leaderboard = pd.read_sql(query3, conn)

if selected_domain != "All":
    leaderboard = leaderboard[
        leaderboard["domain"] == selected_domain
    ]

leaderboard = leaderboard.head(10)

st.dataframe(
    leaderboard,
    use_container_width=True
)

conn.close()

Overwriting dashboard.py


In [76]:
# Query 1: Counts completed interns grouped by domain

query1 = """
SELECT
    domain,
    COUNT(*) AS completed_count
FROM interns
WHERE completion_status = 'Completed'
GROUP BY domain
ORDER BY completed_count DESC;
"""

In [77]:
# Query 2: Calculates the average hackathon score for each domain

query2 = """
SELECT
    domain,
    ROUND(AVG(score),2) AS avg_score
FROM hackathon_scores
GROUP BY domain
ORDER BY avg_score DESC;
"""

In [78]:
# Query 3: Lists interns who scored 85 or above

query3 = """
SELECT
    i.intern_id,
    i.domain,
    h.score
FROM interns i
JOIN hackathon_scores h
ON i.intern_id = h.intern_id
WHERE h.score >= 85
ORDER BY h.score DESC;
"""

In [79]:
# Query 4: Calculates the applicant-to-completed-intern conversion rate for each domain

query4 = """
SELECT
    a.domain,
    COUNT(DISTINCT a.applicant_id) AS total_applicants,
    COUNT(DISTINCT i.intern_id) AS total_completed,
    ROUND(
        100.0 * COUNT(DISTINCT i.intern_id) /
        COUNT(DISTINCT a.applicant_id),2
    ) AS conversion_rate_pct
FROM applicants a
LEFT JOIN interns i
ON a.applicant_id = i.applicant_id
AND i.completion_status='Completed'
GROUP BY a.domain
ORDER BY conversion_rate_pct DESC;
"""

In [80]:
# Query 5: Finds the average hackathon score of interns from each university

query5 = """
SELECT
    a.university,
    ROUND(AVG(h.score),2) AS average_score,
    COUNT(h.intern_id) AS total_interns
FROM applicants a
JOIN interns i
ON a.applicant_id = i.applicant_id
JOIN hackathon_scores h
ON i.intern_id = h.intern_id
GROUP BY a.university
ORDER BY average_score DESC;
"""